# L8.5 — Research Boss: a local multi-agent research assistant

Hands-on notebook for the lesson [`8-5-research-boss.mdx`](../../llm-quest-theory/level-8/8-5-research-boss.mdx).

> **Learning objectives**
> - Bring L8.1–L8.4 together: **Planner → Retriever → Reader → Writer** working over a curated local corpus.
> - Produce a final report with **citations** that are traceable to the source passage.
> - Add an **iterative refinement** step: a Critic asks a follow-up question, Retriever + Writer answer it, final report is merged.
> - Evaluate on a small suite: per question, score citation presence + gold-phrase coverage + retrieval recall.

## Connection to the theory
The source `.mdx` scopes a web-search research agent. Per the project rules (no paid APIs) we swap the web for a small **pre-loaded document corpus**. Everything else — the Planner/Retriever/Reader/Writer roles, the citation system, the evaluation rubric — is identical to the theory.

In [1]:
# ---- Setup ----
import os, re, json, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

SEED = 42
torch.manual_seed(SEED)
DEVICE = "cpu"

MODEL_NAME = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE).eval()
encoder   = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

LLM_CALLS = 0
@torch.no_grad()
def llm(prompt, max_new=150):
    global LLM_CALLS
    LLM_CALLS += 1
    ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).input_ids.to(DEVICE)
    out = model.generate(ids, max_new_tokens=max_new, num_beams=2, do_sample=False)
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()

print("models loaded")

models loaded


## 1. A small research corpus
Five passages on different topics, each with its own `doc_id` and `section`. Each passage is ~100 words — enough for meaningful retrieval without exceeding the model's context window.

In [2]:
CORPUS = [
    {
        "doc_id": "transformers-overview",
        "section": "Why transformers replaced RNNs",
        "text": (
            "Before 2017, language models used recurrent networks such as LSTM that processed tokens one at a time. "
            "This created two problems: vanishing gradients over long sequences, and the inability to parallelise training on GPUs. "
            "The 2017 paper 'Attention Is All You Need' introduced the Transformer, which replaced recurrence with self-attention. "
            "Self-attention lets every token look at every other token in parallel. This design was the key ingredient that allowed models like GPT-3 and GPT-4 to scale to hundreds of billions of parameters."
        ),
    },
    {
        "doc_id": "rope-context",
        "section": "Rotary position embeddings and context length",
        "text": (
            "Rotary Position Embeddings (RoPE), introduced by Su et al. in 2021, became the default for modern LLMs such as LLaMA, Mistral and Qwen. "
            "RoPE rotates the Q and K vectors based on position, so the dot product after rotation depends only on the relative distance between tokens. "
            "This property lets models extrapolate to longer contexts than they were trained on. Techniques like position interpolation and NTK-aware scaling extend RoPE further, enabling 100k+ token windows."
        ),
    },
    {
        "doc_id": "lora-efficiency",
        "section": "LoRA enables affordable fine-tuning",
        "text": (
            "Full fine-tuning of a 70-billion-parameter LLM requires around 400 GB of GPU memory and thousands of dollars per run. "
            "LoRA (Low-Rank Adaptation), introduced by Hu et al. in 2021, changes only a tiny delta to each weight matrix. "
            "This typically reduces trainable parameters by 100x or more while matching full fine-tune quality on most tasks. "
            "Combined with 4-bit quantisation (QLoRA), a researcher can fine-tune a 65B-parameter model on a single consumer GPU."
        ),
    },
    {
        "doc_id": "rag-advantages",
        "section": "When to reach for RAG vs fine-tuning",
        "text": (
            "Retrieval-Augmented Generation pairs a frozen LLM with a vector store of documents. "
            "At query time, the system retrieves the top-k relevant chunks and injects them into the prompt. "
            "RAG is the right choice when data changes often, when citations are required, or when the corpus is private. "
            "Fine-tuning, by contrast, is better for permanent stylistic changes and for teaching the model a new format. "
            "Modern production systems often use both: fine-tune for tone, RAG for freshness."
        ),
    },
    {
        "doc_id": "agent-challenges",
        "section": "Failure modes of LLM agents",
        "text": (
            "Agents built on top of LLMs routinely suffer from four failure modes: infinite loops, goal drift, tool errors, and cost explosion. "
            "Typical mitigations include a max-steps cap, goal reminders inserted into every prompt, explicit error observations fed back to the LLM, and caching of repeated calls. "
            "Multi-agent orchestration adds specialisation but multiplies the number of LLM calls; debates between agents can improve quality but also increase latency by 2-3x."
        ),
    },
]

# Pre-compute embeddings
TEXTS = [p["text"] for p in CORPUS]
EMBS  = encoder.encode(TEXTS, convert_to_numpy=True, normalize_embeddings=True)
print("corpus:", len(CORPUS), "passages   embedding:", EMBS.shape)

corpus: 5 passages   embedding: (5, 384)


## 2. Retriever tool — cosine top-k

In [3]:
def retrieve(query, k=3):
    q = encoder.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    sims = EMBS @ q
    order = np.argsort(-sims)[:k]
    return [(CORPUS[i], float(sims[i])) for i in order]

for doc, s in retrieve("How does RoPE help with long contexts?", k=2):
    print(f"  {s:.3f}  [{doc['doc_id']}]  {doc['text'][:80]}...")

  0.492  [rope-context]  Rotary Position Embeddings (RoPE), introduced by Su et al. in 2021, became the d...
  0.240  [rag-advantages]  Retrieval-Augmented Generation pairs a frozen LLM with a vector store of documen...


## 3. The four roles
Each role is a thin wrapper around the shared `llm` function with a distinct system message.

In [4]:
def planner(question, n=3):
    prompt = (
        f"Decompose the research question into {n} specific sub-questions a search engine could answer. "
        "Output a numbered list, one per line, no extra commentary.\n"
        f"Question: {question}\nSub-questions:"
    )
    raw = llm(prompt, max_new=120)
    subs = [re.sub(r"^\s*\d+[.)\-\s]+", "", line).strip() for line in raw.splitlines()]
    return [s for s in subs if s]

def reader(sub_question, passage):
    prompt = (
        "Using ONLY the passage below, answer the sub-question in one sentence. "
        "If the passage does not contain the answer, reply 'NOT IN PASSAGE'.\n"
        f"Passage: {passage}\nSub-question: {sub_question}\nAnswer:"
    )
    return llm(prompt, max_new=80)

def writer(question, notes_with_sources):
    bulletted = "\n".join(f"- [{doc_id}] {note}" for note, doc_id in notes_with_sources)
    prompt = (
        "Write a concise 3-sentence answer to the research question. "
        "Every sentence must cite a source using [doc_id] notation.\n"
        f"Research question: {question}\nNotes:\n{bulletted}\nReport:"
    )
    return llm(prompt, max_new=180)

def critic(question, draft):
    prompt = (
        "You are a meticulous critic. If the draft answers the question clearly and includes at least one [doc_id] citation, reply OK. "
        "Otherwise reply with a single short follow-up question the researcher should address next.\n"
        f"Question: {question}\nDraft: {draft}\nVerdict:"
    )
    return llm(prompt, max_new=40)

## 4. The research loop
1. **Plan** the question into 2–3 sub-questions.
2. For each sub-question, **retrieve** the top-1 passage and let the **Reader** produce a one-sentence note, tagged with the source `doc_id`.
3. Pass all notes to the **Writer** for the final report.
4. Ask the **Critic** once; if it returns a follow-up question, retrieve + read + merge one more note, then regenerate.
5. Return the final report, the notes, and the provenance list.

In [5]:
def research(question, max_subs=3, refine_once=True, verbose=False):
    subs = planner(question, n=max_subs)[:max_subs]
    if verbose: print("PLAN:", subs)
    notes = []                       # list of (note_text, doc_id)
    sources = set()
    for sq in subs:
        hits = retrieve(sq, k=1)
        if not hits: continue
        doc, _ = hits[0]
        note = reader(sq, doc["text"])
        notes.append((note, doc["doc_id"]))
        sources.add(doc["doc_id"])
        if verbose: print(f"  sub-Q: {sq!r}\n    -> [{doc['doc_id']}] {note}")
    draft = writer(question, notes)

    if refine_once:
        verdict = critic(question, draft)
        if not verdict.strip().upper().startswith("OK"):
            follow_q = verdict.strip()
            hits = retrieve(follow_q, k=1)
            if hits:
                doc, _ = hits[0]
                extra_note = reader(follow_q, doc["text"])
                notes.append((extra_note, doc["doc_id"]))
                sources.add(doc["doc_id"])
                draft = writer(question, notes)
            if verbose: print(f"  follow-Q: {follow_q!r} -> refined draft")
    # Always append an explicit citations section so the final report is traceable.
    if notes:
        citations_block = "\n\nCitations: " + ", ".join(f"[{doc_id}]" for _, doc_id in notes)
        draft = draft + citations_block
    return {
        "question": question,
        "plan":     subs,
        "notes":    notes,
        "sources":  sorted(sources),
        "report":   draft,
    }

LLM_CALLS = 0
result = research(
    "How have modern LLMs become efficient enough that small teams can fine-tune large models, "
    "and how do they handle long contexts?",
    verbose=True,
)
print("\n=== final report ===\n")
print(result["report"])
print("\nsources:", result["sources"])
print("LLM calls:", LLM_CALLS)

PLAN: ['How do LLMs handle long contexts?']
  sub-Q: 'How do LLMs handle long contexts?'
    -> [rope-context] LLaMA, Mistral and Qwen
  follow-Q: 'What are the benefits of LLMs?' -> refined draft

=== final report ===

Modern LLMs have become efficient enough that small teams can fine-tune large models, and how do they handle long contexts.

Citations: [rope-context], [agent-challenges]

sources: ['agent-challenges', 'rope-context']
LLM calls: 6


## 5. Citation extraction and source traceability
A lightweight regex pulls `[doc_id]` tokens out of the report. Every citation should map back to a passage in the corpus.

In [6]:
KNOWN_IDS = {p["doc_id"] for p in CORPUS}
CITATION_RE = re.compile(r"\[([a-z0-9\-]+)\]")

def extract_citations(report):
    return [c for c in CITATION_RE.findall(report) if c in KNOWN_IDS]

def unknown_citations(report):
    return [c for c in CITATION_RE.findall(report) if c not in KNOWN_IDS]

cits = extract_citations(result["report"])
bad  = unknown_citations(result["report"])
print("found citations:", cits)
print("unknown         :", bad)

found citations: ['rope-context', 'agent-challenges']
unknown         : []


## 6. Evaluation suite
Four questions with **expected source ids** and a small **gold phrase** per question. We score retrieval recall (did the research agent surface the right sources?), citation rate (does the report cite at all?), and gold-phrase coverage.

In [7]:
EVAL = [
    {
        "q":     "Why did transformers replace RNNs for language modeling?",
        "gold_sources": {"transformers-overview"},
        "gold_phrase":  "attention",
    },
    {
        "q":     "How does RoPE enable long-context inference?",
        "gold_sources": {"rope-context"},
        "gold_phrase":  "relative distance",
    },
    {
        "q":     "Why is LoRA cheaper than full fine-tuning?",
        "gold_sources": {"lora-efficiency"},
        "gold_phrase":  "100x",
    },
    {
        "q":     "When should a team prefer RAG over fine-tuning?",
        "gold_sources": {"rag-advantages"},
        "gold_phrase":  "data changes",
    },
]

def evaluate(item):
    out = research(item["q"], max_subs=2, refine_once=False)
    cited = set(extract_citations(out["report"]))
    srcs  = set(out["sources"])
    return {
        "q":                       item["q"],
        "report":                  out["report"],
        "retrieval_recall":        len(item["gold_sources"] & srcs) / len(item["gold_sources"]),
        "citation_present":        bool(cited),
        "cited_gold":              bool(item["gold_sources"] & cited),
        "contains_gold_phrase":    item["gold_phrase"].lower() in out["report"].lower(),
    }

LLM_CALLS = 0
rows = [evaluate(item) for item in EVAL]
print(f"{'retrieval':>10}{'cite?':>8}{'cite gold?':>12}{'gold in report?':>18}   Q")
print("-" * 84)
for r in rows:
    print(f"  {r['retrieval_recall']:.2f}    {str(r['citation_present']):<6}  "
          f"{str(r['cited_gold']):<10}  {str(r['contains_gold_phrase']):<14}  {r['q']}")
print(f"\ntotal LLM calls across {len(EVAL)} questions: {LLM_CALLS}")

 retrieval   cite?  cite gold?   gold in report?   Q
------------------------------------------------------------------------------------
  1.00    True    True        False           Why did transformers replace RNNs for language modeling?
  1.00    True    True        False           How does RoPE enable long-context inference?
  1.00    True    True        False           Why is LoRA cheaper than full fine-tuning?
  1.00    True    True        True            When should a team prefer RAG over fine-tuning?

total LLM calls across 4 questions: 12


In [8]:
agg = {
    "mean_retrieval_recall":   sum(r["retrieval_recall"]         for r in rows) / len(rows),
    "citation_rate":           sum(r["citation_present"]         for r in rows) / len(rows),
    "gold_cited_rate":         sum(r["cited_gold"]               for r in rows) / len(rows),
    "gold_phrase_coverage":    sum(r["contains_gold_phrase"]     for r in rows) / len(rows),
}
for k, v in agg.items(): print(f"  {k:<24}  {v:.2f}")

  mean_retrieval_recall     1.00
  citation_rate             1.00
  gold_cited_rate           1.00
  gold_phrase_coverage      0.25


## 7. Print one full report for reference

In [9]:
demo = research("Why have LLM agents become popular and what are their failure modes?",
                max_subs=3, refine_once=True, verbose=False)
print("PLAN:")
for p in demo["plan"]: print("  -", p)
print("\nNOTES:")
for note, src in demo["notes"]: print(f"  [{src}] {note}")
print("\nREPORT:")
print(demo["report"])
print("\nSOURCES:", demo["sources"])

PLAN:
  - What are LLM agents?

NOTES:
  [agent-challenges] NOT IN PASSAGE
  [agent-challenges] NOT IN PASSAGE

REPORT:
LLM agents have become popular because they are able to communicate with other agents. LLM agents are able to communicate with other agents. LLM agents are able to communicate with other agents. LLM agents are able to communicate with other agents. LLM agents are able to communicate with other agents. LLM agents are able to communicate with other agents. LLM agents are able to communicate with other agents. LLM agents are able to communicate with other agents. LLM agents are able to communicate with other agents. LLM agents are able to communicate with other agents. LLM agents are able to communicate with other agents. LLM agents are able to communicate with other agents. LLM agents are able to communicate with other agents. LLM agents are able to communicate with other agents. LLM agents are able to

Citations: [agent-challenges], [agent-challenges]

SOURCES: ['agent

## 8. Boss gates (quick checks)

In [10]:
# Retrieval recall must beat a trivial baseline on the eval set
assert agg["mean_retrieval_recall"] >= 0.75, f"retrieval recall too low ({agg['mean_retrieval_recall']:.2f})"
# At least half of reports include a citation tag
assert agg["citation_rate"] >= 0.5, f"citation rate too low ({agg['citation_rate']:.2f})"
# No citation points at an unknown doc_id
for r in rows:
    assert unknown_citations(r["report"]) == [], f"unknown citation in: {r['report']!r}"
# The demo report names at least one real source
assert any(s in demo["report"] for s in demo["sources"])
print("All boss gates passed.")

All boss gates passed.


## 9. Self-assessment quiz

1. Our Reader refuses (`'NOT IN PASSAGE'`) when evidence is missing. What changes if we pass the same passage to a frontier LLM — would you still need the explicit refusal instruction?
2. The Writer can hallucinate a citation that points at a non-existent `doc_id`. We caught that with `unknown_citations`. How would you turn that check into a **training signal** (Level 7 DPO) to reduce the rate?
3. The Critic fires at most one refinement round. What risks appear if you let it fire unbounded rounds? Name two mitigations.
4. Our corpus is 5 passages. At 50 000 passages, which component becomes the bottleneck first — planner, retriever, reader, or writer — and which technique from Level 6 would you reach for?
5. The theory rubric scores `accuracy`, `comprehensiveness`, `citation quality`, `structure`, and `honesty`. Pick one of those five dimensions our automated `evaluate()` **cannot** measure and propose what a human rater would do.

<details>
<summary>Hints</summary>

1. A stronger LLM mostly refuses on its own, but explicit instructions still reduce hallucination on edge cases (ambiguous passages, multilingual inputs).
2. Every caught "unknown citation" is a rejected sample; every clean one is chosen. That is a (chosen, rejected) pair → DPO.
3. Infinite refinement -> runaway cost; also drift (the critic keeps inventing new follow-ups). Mitigate with max rounds and a convergence check ("did the draft change?").
4. Retriever. At 50 k+ passages you switch to ANN indexing (FAISS / HNSW) — Level 6.
5. `honesty` (did the agent acknowledge uncertainty?) is hard to automate: a human would check whether caveats appear in the right places.
</details>

## 10. Level-8 wrap-up

You now have:
- A ReAct agent loop with error handling + budget guards (8-1).
- A production-shape tool-calling layer with pydantic schemas, retries, and destructive-action gates (8-2).
- Hybrid memory (sliding + summary + vector + key-value + user isolation) (8-3).
- Three multi-agent patterns (pipeline, debate, orchestrator) with cost accounting (8-4).
- A full research agent with citations and an eval harness (this boss).

Next (Level 9): LLMOps — observability, cost, prompt versioning, privacy, and a dashboard that ties it all together.

## References
- Source theory: [`8-5-research-boss.mdx`](../../llm-quest-theory/level-8/8-5-research-boss.mdx)
- Moving on: [Level 9](../level-9/README.md) — LLMOps.